# 01 — Data pipeline
Download RTLCoder + MG-Verilog, download eval sets (eval-only, never trained on), build the corpus (normalise -> dedup -> contamination check -> tag -> split).

In [ ]:
%run notebooks/00_setup.ipynb

In [ ]:
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/eval', exist_ok=True)

# Fetch RTLCoder (27k+ pairs) and MG-Verilog (11k+ modules x 3 description
# levels) via the `datasets` library, then re-shape each into this repo's
# {instruction, code} jsonl schema before build_corpus.py normalises it further.
!pip install -q datasets
from datasets import load_dataset
import json

rtlcoder = load_dataset("ishorn5/RTLCoder-v1.1", split="train")
with open('data/raw/rtlcoder.jsonl', 'w') as f:
    for row in rtlcoder:
        f.write(json.dumps({"instruction": row.get("Instruction", row.get("instruction")),
                             "code": row.get("Response", row.get("code"))}) + "\n")

mg_verilog = load_dataset("GaTech-EIC/MG-Verilog", split="train")
with open('data/raw/mg_verilog.jsonl', 'w') as f:
    for row in mg_verilog:
        # MG-Verilog ships 3 description granularities per module; use the
        # 'detailed' level as the instruction so difficulty roughly matches RTLCoder
        f.write(json.dumps({"instruction": row.get("description_detailed", row.get("description")),
                             "code": row.get("code")}) + "\n")

print('RTLCoder:', sum(1 for _ in open('data/raw/rtlcoder.jsonl')))
print('MG-Verilog:', sum(1 for _ in open('data/raw/mg_verilog.jsonl')))

In [ ]:
# Eval sets -- EVAL ONLY, never in the training corpus. VerilogEval v2 (156
# problems) and RTLLM v2 (50 designs). Each row needs: id, instruction,
# testbench, top_module, tier (assigned via the tagger against the
# reference solution), code (reference solution -- used only for the
# contamination check, never for scoring).
!git clone --depth 1 https://github.com/NVlabs/verilog-eval /tmp/verilogeval
!git clone --depth 1 https://github.com/hkust-zhiyao/RTLLM /tmp/rtllm
# scripts/build_eval_jsonl.py's GLOB_PATTERNS must match whatever tag you
# just cloned -- both repos have reorganised their layout before, so fix
# the patterns there first if this step reports 0 problem directories.
!python -m scripts.build_eval_jsonl --src /tmp/verilogeval --out data/eval/verilogeval_v2.jsonl --benchmark verilogeval
!python -m scripts.build_eval_jsonl --src /tmp/rtllm --out data/eval/rtllm_v2.jsonl --benchmark rtllm

In [ ]:
!python -m src.data.build_corpus \
  --sources rtlcoder=data/raw/rtlcoder.jsonl mg-verilog=data/raw/mg_verilog.jsonl \
  --eval-sets data/eval/verilogeval_v2.jsonl data/eval/rtllm_v2.jsonl \
  --out artifacts/corpus.jsonl \
  --probe-frac 0.10 --seed 1337

In [ ]:
# Sanity-check tier/split balance before training on it
import json
from collections import Counter
rows = [json.loads(l) for l in open('artifacts/corpus.jsonl')]
print('total:', len(rows))
print('by split:', Counter(r['split'] for r in rows))
print('by tier (train):', Counter(r['tags']['tier'] for r in rows if r['split']=='train'))
print('by tier (probe):', Counter(r['tags']['tier'] for r in rows if r['split']=='probe'))